# Explore the Stock Market Agent

Mirrors `01-agentic-rag/code/agents.ipynb`: call a tool directly, then run the full agent loop, then a multi-turn conversation, then the ToyAIKit-based runner used in `04-evaluation`'s agent-evaluation lesson.

In [ ]:
import sys
sys.path.append('..')

from dotenv import load_dotenv
load_dotenv('../.env')

from agent import agent_loop, build_toyaikit_runner, extract_tool_calls
from tools import get_stock_snapshot, get_market_status

## 1. Call a tool directly

In [ ]:
get_market_status()

In [ ]:
get_stock_snapshot('AAPL')

## 2. Full agent loop for one question
Returns an `AgentCallRecord` (same shape as the course's `LLMCallRecord` in `05-monitoring/code/metrics.py`) plus the updated message history.

In [ ]:
record, messages = agent_loop("How is Tesla stock doing today compared to yesterday's close?")
print(record.answer)
print()
print('tool calls:', record.tool_calls)
print('cost: $%.5f' % record.cost)

## 3. Multi-turn conversation
Pass `messages` back in as `previous_messages` for a follow-up turn.

In [ ]:
record1, messages1 = agent_loop("What's NVIDIA's current price?")
print(record1.answer)

In [ ]:
record2, messages2 = agent_loop("How does that compare to its price a month ago?", previous_messages=messages1)
print(record2.answer)

## 4. ToyAIKit-based interactive runner
Same pattern as `04-evaluation/lessons/14-agent-evaluation.md`: `runner.loop()` returns `.last_message`, `.all_messages`, and `.cost`, and `extract_tool_calls()` pulls the trajectory out of `all_messages`.

In [ ]:
runner, callback = build_toyaikit_runner()
result = runner.loop(prompt="Any recent news on Amazon?", callback=callback)
print(result.cost)
extract_tool_calls(result.all_messages)